# FHIR Gold Resource Snapshot


## Spark SQL resource aggregates


In [ ]:
from pyspark.sql import functions as F
RESOURCES = ('Patient', 'Encounter', 'Observation', 'Condition')

def table_name(layer, resource):
    return f"{layer}.fhir_{resource.lower()}"


def build_gold(spark):
    """Publish a report-ready snapshot with one row per resource type."""
    spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
    metrics = []
    for resource in RESOURCES:
        silver = spark.table(table_name("silver", resource))
        metrics.append(spark.sql("""
            SELECT
                   COUNT(*) AS observed_versions,
                   COUNT_IF(is_current) AS current_resources,
                   MAX(extraction_timestamp) AS latest_extraction_timestamp,
                   MAX(raw_saved_timestamp) AS latest_raw_saved_timestamp,
                   MAX(silver_saved_timestamp) AS latest_silver_saved_timestamp
            FROM {silver}
        """, silver=silver).withColumn("resource_type", F.lit(resource)))
    snapshot = metrics[0]
    for metric in metrics[1:]:
        snapshot = snapshot.unionByName(metric)
    (snapshot.withColumn("gold_saved_timestamp", F.current_timestamp())
     .write.format("delta").mode("overwrite").saveAsTable("gold.fhir_resource_snapshot"))


In [ ]:
spark.conf.set('spark.sql.session.timeZone', 'UTC')
build_gold(spark)
